# Day 5-1: HaGRID 손 제스처 데이터 EDA

**강의 시간**: 1.5시간  
**학습 목표**:
- HaGRID 데이터셋 탐색
- 19가지 제스처 이해
- 실시간 인식 요구사항 학습
- 경량 모델의 필요성 인식

**데이터셋**: HaGRID (Hand Gesture Recognition Image Dataset)  
**크기**: ~3.8GB (153,735개 JPEG)  
**클래스**: 19개 (18개 제스처 + no_gesture)

## 🔧 0. 환경 설정

In [ ]:
# 라이브러리 설치
%pip install -q 'mlflow>=2,<3' dagshub tensorflow opencv-python scikit-learn

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras

import mlflow
import dagshub

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow {tf.__version__}")
print(f"✅ GPU: {len(tf.config.list_physical_devices('GPU'))} devices")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm
font_path = "NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# MLflow 설정
import mlflow
import dagshub

# 🔥 본인의 정보로 수정!
repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name  = # 🔥 직접 작성이 필요합니다.

dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
mlflow.set_experiment('day5-gesture-recognition')

print("✅ MLflow 설정 완료!")

## 📥 1. HaGRID 데이터 다운로드

### HaGRID 512p 다운로드

Kaggle에서 다운로드합니다. Colab 보안 비밀에 `KAGGLE_USERNAME`과 `KAGGLE_API_TOKEN`이 설정되어 있어야 합니다.

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    dataset_id = 'innominate817/hagrid-classification-512p-no-gesture-150k'
    print(f"📥 {dataset_id} 다운로드 시작...")

    api.dataset_download_files(
        dataset_id,
        path='./data',
        unzip=True,
        quiet=False
    )

    print("\n✅ 다운로드 및 압축 해제 완료!")

except Exception as e:
    print(f"\n❌ 오류 발생: {e}")

## 🔍 2. 데이터 구조 탐색

In [ ]:
# 데이터 구조 확인
data_dir = Path('./data/hagrid-classification-512p-no-gesture-150k')

print("="*60)
print("  데이터셋 구조 확인")
print("="*60)

if data_dir.exists():
    print(f"✅ 데이터 폴더 발견: {data_dir}")

    gesture_folders = sorted([d for d in data_dir.iterdir() if d.is_dir()])
    gesture_names = [f.name for f in gesture_folders]

    print(f"\n클래스 수: {len(gesture_names)}")
    print(f"\n제스처 목록:")
    for i, name in enumerate(gesture_names, 1):
        print(f"  {i:2d}. {name}")
else:
    print("❌ 데이터 폴더를 찾을 수 없습니다!")

In [ ]:
# 클래스별 이미지 수 확인
gesture_counts = {}

for gesture_folder in gesture_folders:
    num_images = len([f for f in gesture_folder.iterdir()
                     if f.is_file() and f.suffix.lower() in ['.jpg', '.jpeg', '.png']])
    gesture_counts[gesture_folder.name] = num_images
    print(f"{gesture_folder.name:20s}: {num_images:6d} images")

total = sum(gesture_counts.values())
print("="*60)
print(f"{'Total':20s}: {total:6d} images")
print("="*60)

In [ ]:
# 클래스 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sorted_gestures = sorted(gesture_counts.items(), key=lambda x: x[1], reverse=True)
names, counts = zip(*sorted_gestures)
colors = plt.cm.tab20(np.linspace(0, 1, len(names)))

bars = axes[0].barh(range(len(names)), counts, color=colors, edgecolor='black')
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=9)
axes[0].set_xlabel('이미지 수', fontweight='bold', fontsize=12)
axes[0].set_title('제스처별 이미지 분포', fontweight='bold', fontsize=14)
axes[0].grid(axis='x', alpha=0.3)

for i, (bar, count) in enumerate(zip(bars, counts)):
    axes[0].text(count + 100, i, f'{count:,}', va='center', fontsize=8)

top_10 = dict(sorted_gestures[:10])
axes[1].pie(top_10.values(), labels=top_10.keys(), autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 9})
axes[1].set_title('상위 10개 제스처 비율', fontweight='bold', fontsize=14)

plt.suptitle(f'HaGRID Classification 512p — {len(gesture_names)}개 제스처',
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

min_count = min(gesture_counts.values())
max_count = max(gesture_counts.values())
balance_ratio = min_count / max_count
print(f"\n균형도: {min_count:,} / {max_count:,} = {balance_ratio:.3f}")
print("✅ 클래스가 잘 균형잡혀 있습니다!" if balance_ratio > 0.8 else "⚠️ 클래스 불균형 있음")

## 🖼️ 3. 샘플 이미지 시각화

In [ ]:
# 각 제스처별 샘플 이미지 (5×4 grid)
sample_gestures = gesture_names[:20]

fig, axes = plt.subplots(5, 4, figsize=(16, 20))
axes = axes.flatten()

for i, gesture_name in enumerate(sample_gestures):
    if i >= len(axes):
        break

    gesture_folder = data_dir / gesture_name
    image_paths = list(gesture_folder.glob('*.jpeg'))[:1]

    if len(image_paths) > 0:
        img = cv2.imread(str(image_paths[0]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        axes[i].set_title(f'{gesture_name}\n{img.shape[0]}×{img.shape[1]}',
                         fontweight='bold', fontsize=11)
        axes[i].axis('off')

for i in range(len(sample_gestures), len(axes)):
    axes[i].axis('off')

plt.suptitle('HaGRID 손 제스처 샘플 이미지', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 💾 4. 데이터 로딩 & 전처리

In [ ]:
# 전체 경로 수집
all_paths = []
all_labels = []

gesture_to_idx = {name: idx for idx, name in enumerate(gesture_names)}

print("📋 파일 경로 수집 중...")

for gesture_name in gesture_names:
    gesture_folder = data_dir / gesture_name
    image_paths = list(gesture_folder.glob('*.jpeg'))
    for img_path in image_paths:
        all_paths.append(str(img_path))
        all_labels.append(gesture_to_idx[gesture_name])
    print(f"  {gesture_name:20s}: {len(image_paths):6d}개")

print(f"\n✅ 총 {len(all_paths):,}개 이미지")

# Train/Val/Test Split (70/15/15)
from sklearn.model_selection import train_test_split

# 🔥 직접 작성이 필요합니다. (Train 70%, Temp 30% 분리 — test_size=0.3, stratify=all_labels, random_state=42)
train_paths, temp_paths, train_labels, temp_labels = # 🔥 직접 작성이 필요합니다.

# 🔥 직접 작성이 필요합니다. (Temp를 Val/Test 50/50으로 분리 — test_size=0.5, stratify=temp_labels, random_state=42)
val_paths, test_paths, val_labels, test_labels = # 🔥 직접 작성이 필요합니다.

print("\n" + "="*60)
print("  데이터 분할 (Train/Val/Test)")
print("="*60)
print(f"Train: {len(train_paths):6d}개 ({len(train_paths)/len(all_paths)*100:.1f}%)")
print(f"Val  : {len(val_paths):6d}개 ({len(val_paths)/len(all_paths)*100:.1f}%)")
print(f"Test : {len(test_paths):6d}개 ({len(test_paths)/len(all_paths)*100:.1f}%)")
print(f"Total: {len(all_paths):6d}개")
print("="*60)

## 📊 5. tf.data.Dataset 생성 (메모리 효율적)

In [ ]:
# Preprocessing 함수
def load_and_preprocess(path, label, target_size=(224, 224)):
    """이미지 로드 & 전처리"""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)   # JPEG, RGB 3채널
    img = tf.image.resize(img, target_size)        # 512×512 → 224×224
    img = img / 255.0                              # 0~1 정규화
    return img, label

print("✅ Preprocessing 함수 정의 완료!")
print("   512×512 → 224×224 리사이즈")
print("   Day 4(PNG, Grayscale)와 달리 JPEG + RGB → decode_jpeg + channels=3")

In [ ]:
# tf.data.Dataset 생성
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_dataset = train_dataset.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
train_dataset = train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_dataset = val_dataset.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("✅ Dataset 생성 완료!")
print(f"   Train batches: {len(train_dataset)}")
print(f"   Val batches: {len(val_dataset)}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"\n💡 메모리 효율: 배치 단위로만 로드 (~19MB/batch)")

## 🏗️ 6. Baseline CNN 모델

In [ ]:
from tensorflow.keras import layers, Model

In [ ]:
# Baseline CNN (19개 클래스)
def build_baseline_cnn(input_shape=(224, 224, 3), num_classes=19):
    """Baseline CNN for Gesture Recognition (19 classes)"""

    model = keras.Sequential([
        # Conv Block 1: Conv(32) + MaxPool + BN + Dropout(0.25)
        # 🔥 직접 작성이 필요합니다.

        # Conv Block 2: Conv(64) + MaxPool + BN + Dropout(0.25)
        # 🔥 직접 작성이 필요합니다.

        # Conv Block 3: Conv(128) + MaxPool + BN + Dropout(0.25)
        # 🔥 직접 작성이 필요합니다.

        # Classifier
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='Baseline_CNN')

    return model

baseline_model = build_baseline_cnn(num_classes=len(gesture_names))

print("✅ Baseline CNN 모델 생성 완료!")
print(f"   Input: 224×224×3")
print(f"   Output: {len(gesture_names)} classes")
print(f"   Parameters: {baseline_model.count_params():,}")

In [ ]:
# 모델 컴파일
baseline_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ 모델 컴파일 완료!")

## 🏃 7. Baseline 모델 학습

In [ ]:
# Callbacks
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

print("✅ Callbacks 설정 완료!")

In [ ]:
# MLflow 실험
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다. (예: 'Baseline_CNN_HaGRID')
    mlflow.log_params({
        'model': 'Baseline_CNN',
        'dataset': 'HaGRID_512p_150k',
        'num_classes': len(gesture_names),
        'input_size': '224x224',
        'optimizer': 'adam',
        'epochs': 15,
        'batch_size': BATCH_SIZE
    })

    print("🏃 모델 학습 시작...\n")
    # ⏱️ GPU에 따라 20~30분 소요

    history = baseline_model.fit(
        train_dataset,
        epochs=15,
        validation_data=val_dataset,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    final_loss, final_acc = baseline_model.evaluate(val_dataset, verbose=0)

    mlflow.log_metrics({
        'final_val_loss': final_loss,
        'final_val_accuracy': final_acc
    })

    mlflow.keras.log_model(baseline_model, 'model')

    print(f"\n{'='*60}")
    print("  Baseline 모델 학습 완료")
    print('='*60)
    print(f"  Val Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)")
    print('='*60)

## 📈 8. 모델 평가

In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('학습 곡선 - Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Accuracy', fontweight='bold')
axes[1].set_title('학습 곡선 - Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Baseline CNN 학습 곡선', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Validation 예측
print("🔮 Validation set 예측 중...")

y_true = []
y_pred = []

for images, labels in val_dataset:
    preds = baseline_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("✅ 예측 완료!")
print(f"   Accuracy: {np.mean(y_true == y_pred):.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Confusion Matrix (상위 10개)
cm = confusion_matrix(y_true, y_pred)

top_10_indices = sorted(range(len(gesture_names)),
                       key=lambda i: np.sum(cm[i]), reverse=True)[:10]
top_10_names = [gesture_names[i] for i in top_10_indices]
cm_top10 = cm[np.ix_(top_10_indices, top_10_indices)]

plt.figure(figsize=(12, 10))
sns.heatmap(cm_top10, annot=True, fmt='d', cmap='Blues',
            xticklabels=top_10_names, yticklabels=top_10_names)
plt.xlabel('예측', fontweight='bold', fontsize=12)
plt.ylabel('실제', fontweight='bold', fontsize=12)
plt.title('Confusion Matrix — Baseline CNN (Top 10)',
          fontweight='bold', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
report = classification_report(y_true, y_pred,
                              target_names=gesture_names,
                              digits=4)
print("="*80)
print("  Classification Report")
print("="*80)
print(report)
print("="*80)

## ⚡ 9. 모델 크기 & 추론 속도

In [ ]:
# 모델 크기 & 추론 속도 측정
import time

baseline_model.save('baseline_hagrid.keras')
model_size = os.path.getsize('baseline_hagrid.keras') / (1024**2)

print("="*60)
print("  모델 성능 지표")
print("="*60)
print(f"Model Size: {model_size:.2f} MB")
print(f"Parameters: {baseline_model.count_params():,}")
print(f"Val Accuracy: {final_acc*100:.2f}%")

# Warmup
for images, _ in val_dataset.take(1):
    _ = baseline_model.predict(images, verbose=0)

# 추론 속도 측정
latencies = []
for images, _ in val_dataset.take(10):
    start = time.time()
    _ = baseline_model.predict(images, verbose=0)
    latency = (time.time() - start) * 1000 / BATCH_SIZE
    latencies.append(latency)

avg_latency = np.mean(latencies)
print(f"Avg Latency: {avg_latency:.2f} ms/image")
print(f"Throughput: {1000/avg_latency:.1f} images/sec")
print("="*60)

print(f"\n💡 목표:")
print(f"   Size: < 15MB ({'✅' if model_size < 15 else '❌'})")
print(f"   Latency: < 50ms ({'✅' if avg_latency < 50 else '❌'})")
print(f"   Accuracy: > 80% ({'✅' if final_acc > 0.80 else '❌'})")

## ✅ Day 5-1 완료 체크리스트

- [ ] HaGRID Classification 512p 다운로드 (~3.8GB)
- [ ] 19개 제스처 클래스 폴더 구조 확인
- [ ] 클래스 분포 시각화 (균형 확인 ~8,500개/클래스)
- [ ] 제스처별 샘플 이미지 시각화 (5×4 그리드)
- [ ] Train/Val/Test Split (70/15/15, stratify 적용)
- [ ] tf.data.Dataset 파이프라인 구축
- [ ] Baseline CNN 구현 (3개 Conv Block)
- [ ] 모델 학습 (MLflow run_name 기록)
- [ ] Confusion Matrix 확인
- [ ] 모델 크기 & 추론 속도 측정
- [ ] 목표: Val Accuracy ~73%, Size <15MB, Latency <50ms

## 🎯 다음 단계 (Day 5-2)

**MobileNetV2 Transfer Learning**
- ImageNet Pretrained MobileNetV2
- Depthwise Separable Convolution
- Data Augmentation
- Fine-tuning

**목표**: Val Accuracy 90%+, Size < 15MB, Latency < 30ms